In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/dfgdataset2/dataset_graphcodebert.jsonl
/kaggle/input/model2/model.bin


In [2]:
!pip install torch transformers tree_sitter==0.21.3 scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.2/502.2 kB 12.0 MB/s eta 0:00:00


In [3]:
# model.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss

class Model(nn.Module):   
    def __init__(self, encoder, config, tokenizer, args):
        super(Model, self).__init__()
        self.encoder = encoder
        self.config = config
        self.tokenizer = tokenizer
        self.args = args
        
        # Classification Head
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, 2) # 0 or 1

    def forward(self, input_ids=None, p_ids=None, attn_mask=None, labels=None): 
        # Source Code: input_ids
        # Position info: p_ids
        # PRE-CALCULATED MASK: attn_mask
        
        if input_ids is not None:
            # 1. Prepare Mask (Float, 4D)
            # attn_mask is 1 for attend, 0 for mask.
            # We want 0 for attend, large negative for mask.
            extended_attention_mask = (1.0 - attn_mask) * -10000.0
            extended_attention_mask = extended_attention_mask.unsqueeze(1)
            # Shape: [Batch, 1, Seq, Seq]

            # 2. Get Embeddings (Words + Positions)
            # We call the internal embedding layer directly to combine input_ids and p_ids
            embedding_output = self.encoder.embeddings(
                input_ids=input_ids, 
                position_ids=p_ids
            )

            # 3. Pass through Encoder layers (Bypass Model wrapper to skip mask checks)
            # We call the internal 'encoder' directly with our custom mask
            encoder_outputs = self.encoder.encoder(
                embedding_output,
                attention_mask=extended_attention_mask,
                head_mask=[None] * self.config.num_hidden_layers
            )
            
            sequence_output = encoder_outputs[0]
            
            # 4. Classification
            # Take the [CLS] token (index 0)
            logits = self.classifier(self.dropout(sequence_output[:, 0, :]))
            prob = F.softmax(logits, dim=-1)

            if labels is not None:
                loss_fct = CrossEntropyLoss()
                loss = loss_fct(logits, labels)
                return loss, prob
            else:
                return prob

In [4]:
# train.py
import os
import json
import torch
import logging
import random
import numpy as np
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import DataLoader, Dataset, RandomSampler, SequentialSampler
from torch.optim import AdamW
from transformers import (get_linear_schedule_with_warmup, 
                          RobertaConfig, RobertaModel, RobertaTokenizerFast)
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
from tree_sitter import Language, Parser

# --- CONFIGURATION ---
class Args:
    output_dir = "saved_models"
    model_name_or_path = "microsoft/graphcodebert-base"
    config_name = ""
    tokenizer_name = "microsoft/graphcodebert-base"
    
    # Data Paths
    train_file = "/kaggle/input/dfgdataset2/dataset_graphcodebert.jsonl" # Your output file
    
    # Params
    code_length = 384     # Max code tokens
    data_flow_length = 128 # Max DFG nodes
    train_batch_size = 16
    eval_batch_size = 32
    learning_rate = 2e-5
    max_grad_norm = 1.0
    num_train_epochs = 3
    seed = 42
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    n_gpu = torch.cuda.device_count()

args = Args()
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)


# --- TRAINING FUNCTIONS ---
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if args.n_gpu > 0:
        torch.cuda.manual_seed_all(seed)

def train(model, tokenizer):
    dataset = TextDataset(tokenizer, args, args.train_file)
    
    # Split Train/Val (90/10)
    train_size = int(0.9 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])
    
    train_sampler = RandomSampler(train_dataset)
    train_dataloader = DataLoader(train_dataset, sampler=train_sampler, batch_size=args.train_batch_size, num_workers=2,pin_memory=True)
    
    # Optimizer
    optimizer = AdamW(model.parameters(), lr=args.learning_rate, eps=1e-8)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=len(train_dataloader) * args.num_train_epochs)

    scaler = GradScaler()
    
    logger.info("***** Running training *****")
    model.train()
    best_acc = 0.0
    
    for epoch in range(args.num_train_epochs):
        tr_loss = 0
        for step, batch in enumerate(tqdm(train_dataloader, desc=f"Epoch {epoch}")):
            inputs = {
                'input_ids': batch['input_ids'].to(args.device),
                'p_ids': batch['p_ids'].to(args.device),
                'attn_mask': batch['attn_mask'].to(args.device),
                'labels': batch['label'].to(args.device)
            }
            
            optimizer.zero_grad()
            
            # 1. Runs the forward pass with autocasting (FP16)
            with autocast():
                loss, prob = model(**inputs)
            
            # 2. Scales loss and calls backward() to create scaled gradients
            scaler.scale(loss).backward()
            
            # 3. Unscales gradients and updates weights
            scaler.step(optimizer)
            scaler.update()
            
            scheduler.step()
            tr_loss += loss.item()
            
        # Validation
        acc = evaluate(model, val_dataset)
        logger.info(f"Epoch {epoch} | Loss: {tr_loss/len(train_dataloader)} | Val Acc: {acc}")
        
        if acc > best_acc:
            best_acc = acc
            # Save Model
            if not os.path.exists(args.output_dir): os.makedirs(args.output_dir)
            torch.save(model.state_dict(), os.path.join(args.output_dir, "model.bin"))
            logger.info("Saved new best model.")

def evaluate(model, dataset):
    dataloader = DataLoader(dataset, batch_size=args.eval_batch_size)
    model.eval()
    preds = []
    labels = []
    
    for batch in dataloader:
        with torch.no_grad():
            inputs = {
                'input_ids': batch['input_ids'].to(args.device),
                'p_ids': batch['p_ids'].to(args.device),
                'attn_mask': batch['attn_mask'].to(args.device) # <--- CRITICAL FIX: MUST BE INCLUDED
            }
            probs = model(**inputs)
            preds.extend(torch.argmax(probs, dim=-1).cpu().numpy())
            labels.extend(batch['label'].cpu().numpy())
            
    model.train()
    return accuracy_score(labels, preds)



2026-01-03 12:35:19.794137: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767443719.999660      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767443720.054668      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767443720.476748      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767443720.476796      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767443720.476798      23 computation_placer.cc:177] computation placer alr

In [5]:
class SimpleCodeDataset(Dataset):
    def __init__(self, tokenizer, args, file_path):
        self.args = args
        self.tokenizer = tokenizer
        # We load the exact same file
        with open(file_path, 'r') as f:
            self.lines = f.readlines()

    def __len__(self):
        return len(self.lines)

    def __getitem__(self, item):
        line = self.lines[item]
        # Fallback logic for errors
        try:
            entry = json.loads(line)
        except:
            return self.__getitem__((item + 1) % len(self.lines))
            
        code = entry.get('code', '')
        label = int(entry.get('label', 0)) if entry.get('label') is not None else 0

        # Simple Tokenization (Text Only)
        # CodeBERT handles code just like text
        tokens = self.tokenizer(
            code, 
            max_length=self.args.code_length, 
            truncation=True, 
            padding='max_length'
        )

        return {
            'input_ids': torch.tensor(tokens['input_ids'], dtype=torch.long),
            'attention_mask': torch.tensor(tokens['attention_mask'], dtype=torch.long),
            'label': torch.tensor(label, dtype=torch.long)
        }

In [6]:
class SimpleModel(nn.Module):   
    def __init__(self, encoder, config, tokenizer, args):
        super(SimpleModel, self).__init__()
        self.encoder = encoder
        self.config = config
        self.tokenizer = tokenizer
        self.args = args
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, 2) 

    def forward(self, input_ids=None, attention_mask=None, labels=None): 
        # Standard RoBERTa forward pass
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)[0]
        
        # Classification on the [CLS] token (index 0)
        logits = self.classifier(self.dropout(outputs[:, 0, :]))
        prob = F.softmax(logits, dim=-1)

        if labels is not None:
            loss_fct = CrossEntropyLoss()
            loss = loss_fct(logits, labels)
            return loss, prob
        else:
            return prob

In [7]:
class TextDataset(Dataset):
    def __init__(self, tokenizer, args, file_path):
        self.args = args
        self.tokenizer = tokenizer
        self.total_len = args.code_length + args.data_flow_length
        
        # Load raw lines
        with open(file_path, 'r') as f:
            self.lines = f.readlines()
            
    def __len__(self):
        return len(self.lines)

    def _get_char_index(self, code_lines, coord):
        """Converts (row, col) from tree-sitter to absolute char index."""
        row, col = coord
        # Calculate cumulative length of previous lines
        # Note: We assume code_lines preserves newlines or we reconstruct carefully
        char_idx = 0
        for i in range(min(row, len(code_lines))):
            char_idx += len(code_lines[i]) 
        return char_idx + col

    def __getitem__(self, item):
        line = self.lines[item]
        entry = json.loads(line)
        
        code = entry.get('code', '')
        # Ensure we keep the DFG length limit
        dfg = entry.get('dfg', [])[:self.args.data_flow_length]
        label = int(entry.get('label', 0)) if entry.get('label') is not None else 0

        # --- 1. Tokenization with Offsets ---
        # We need offset_mapping to align DFG nodes to tokens
        tokens_obj = self.tokenizer(
            code, 
            max_length=self.args.code_length, 
            truncation=True, 
            padding='max_length',
            return_offsets_mapping=True # CRITICAL FIX
        )
        input_ids = tokens_obj['input_ids']
        offsets = tokens_obj['offset_mapping']
        
        # Split code into lines for coordinate calculation (keeping newlines is safer for indices)
        # We try to mimic how the original code was likely split
        code_lines = code.splitlines(keepends=True)

        # --- 2. Prepare DFG Nodes ---
        # We treat the list of DFGs as the "Graph Nodes" appended to the sequence
        dfg_ids = [self.tokenizer.unk_token_id] * len(dfg)
        
        # Map: DFG Node Index -> List of Code Token Indices it aligns to
        node_to_token_map = {}
        
        # Map: (start_r, start_c, end_r, end_c) -> DFG Node Index
        # This helps us find "parents" in the node list
        pos_to_node_idx = {}

        for node_idx, item in enumerate(dfg):
            # item structure: [var_name, [[start_r, start_c], [end_r, end_c]], "action", [parents], [parent_pos]]
            # Note: Adjust indices if your JSON structure differs slightly
            var_name = item[0]
            start_pos = item[1][0] # [r, c]
            end_pos = item[1][1]   # [r, c]
            
            # Save position for edge linking later
            pos_key = (start_pos[0], start_pos[1], end_pos[0], end_pos[1])
            pos_to_node_idx[pos_key] = node_idx
            
            # Calculate absolute char range
            try:
                char_start = self._get_char_index(code_lines, start_pos)
                char_end = self._get_char_index(code_lines, end_pos)
                
                # Find overlapping tokens
                aligned_tokens = []
                for t_idx, (t_start, t_end) in enumerate(offsets):
                    # Skip special tokens (tuples are (0,0))
                    if t_start == t_end: continue
                    
                    # Check overlap: Token is inside Variable OR Variable is inside Token
                    if (t_start >= char_start and t_end <= char_end) or \
                       (char_start >= t_start and char_end <= t_end):
                        aligned_tokens.append(t_idx)
                
                node_to_token_map[node_idx] = aligned_tokens
            except IndexError:
                # Fallback if coordinates are out of bounds (parsing errors)
                node_to_token_map[node_idx] = []

        # --- 3. Construct Sparse Attention Mask ---
        # Size: (code_len + dfg_len) x (code_len + dfg_len)
        total_len = self.args.code_length + len(dfg) # dynamic based on actual DFG size
        attn_mask = np.zeros((self.total_len, self.total_len), dtype=bool)
        
        # A. Self-Attention for Code (Standard)
        # Code tokens see each other
        c_len = self.args.code_length
        attn_mask[:c_len, :c_len] = True
        
        # B. DFG Structure
        for node_idx, item in enumerate(dfg):
            abs_node_idx = c_len + node_idx
            
            # 1. Node <-> Token Alignment (The "Anchor")
            # The node attends to the tokens it represents, and vice versa
            tokens = node_to_token_map.get(node_idx, [])
            for t_idx in tokens:
                attn_mask[abs_node_idx, t_idx] = True # Node sees Token
                attn_mask[t_idx, abs_node_idx] = True # Token sees Node
            
            # 2. Node -> Node Edges (Data Flow)
            # The node attends to its data-flow parents
            parent_positions = item[4] # List of parent positions
            for p_pos in parent_positions:
                # p_pos is [[r,c], [r,c]]
                p_key = (p_pos[0][0], p_pos[0][1], p_pos[1][0], p_pos[1][1])
                
                if p_key in pos_to_node_idx:
                    parent_idx = pos_to_node_idx[p_key]
                    abs_parent_idx = c_len + parent_idx
                    
                    attn_mask[abs_node_idx, abs_parent_idx] = True # Child sees Parent
                    # Note: GraphCodeBERT usually uses bidirectional or child->parent. 
                    # Adding both helps info flow.
                    attn_mask[abs_parent_idx, abs_node_idx] = True 
            
            # 3. Self Loop (Node sees itself)
            attn_mask[abs_node_idx, abs_node_idx] = True

        # --- 4. Final Assembly ---
        full_input_ids = input_ids + dfg_ids
        
        # Position IDs: Code (0..N) + DFG (Always 0 or special index)
        # GraphCodeBERT typically uses standard positional embeddings for code 
        # and a fixed position (like 0 or max_len) for nodes to indicate they aren't sequential.
        p_ids = [i + 2 for i in range(c_len)] + [0] * len(dfg_ids)

        # Pad everything to strict self.total_len
        curr_len = len(full_input_ids)
        padding_len = self.total_len - curr_len
        
        if padding_len > 0:
            full_input_ids += [self.tokenizer.pad_token_id] * padding_len
            p_ids += [1] * padding_len # Pad pos id
            # attn_mask is already 0s in the padded area, so we are good.
        
        return {
            'input_ids': torch.tensor(full_input_ids, dtype=torch.long),
            'p_ids': torch.tensor(p_ids, dtype=torch.long),
            'attn_mask': torch.tensor(attn_mask, dtype=torch.float),
            'label': torch.tensor(label, dtype=torch.long)
        }

In [8]:
import os
import torch
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm

def manual_evaluation(model, tokenizer, args):
    print("--- Starting Manual Evaluation ---")
    
    # 1. Load the BEST model from disk (Fixes the "Last vs Best" issue)
    model_path = "/kaggle/input/model2/model.bin"
    #model_path = os.path.join(args.output_dir, "model.bin")
    if os.path.exists(model_path):
        print(f"Loading best checkpoint from {model_path}...")
        model.load_state_dict(torch.load(model_path))
    else:
        print("Warning: No saved model found. Evaluating current model state.")

    model.to(args.device)
    model.eval()

    # 2. Reload dataset
    full_dataset = TextDataset(tokenizer, args, args.train_file)
    
    # 3. Re-create split to ensure we evaluate on the Validation set
    generator = torch.Generator().manual_seed(args.seed)
    train_size = int(0.9 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    _, val_dataset = random_split(full_dataset, [train_size, val_size], generator=generator)
    
    print(f"Evaluating on {len(val_dataset)} examples...")
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=args.eval_batch_size, 
        num_workers=2,
        pin_memory=True
    )
    
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Calculating Metrics"):
            inputs = {
                'input_ids': batch['input_ids'].to(args.device),
                'p_ids': batch['p_ids'].to(args.device),
                'attn_mask': batch['attn_mask'].to(args.device) # <--- CRITICAL FIX: UNCOMMENTED
            }
            labels = batch['label'].to(args.device)
            
            probs = model(**inputs)
            preds = torch.argmax(probs, dim=-1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    # Print Report
    print("\n" + "="*30)
    print("FINAL RESULTS (Best Checkpoint)")
    print("="*30)
    
    acc = accuracy_score(all_labels, all_preds)
    print(f"Overall Accuracy: {acc:.4%}")
    print("-" * 30)
    
    print(classification_report(all_labels, all_preds, target_names=['Safe (0)', 'Vulnerable (1)'], digits=4))
    
    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()
    print("\nConfusion Matrix:")
    print(f"True Negatives (Safe correctly identified): {tn}")
    print(f"False Positives (Safe flagged as Vuln):     {fp}")
    print(f"False Negatives (Vuln missed):              {fn}")
    print(f"True Positives (Vuln found):                {tp}")

In [9]:
def train_codebert():
    # 1. Config for Model B
    args.output_dir = "saved_models_codebert" # Different folder
    args.model_name_or_path = "microsoft/codebert-base"
    args.tokenizer_name = "microsoft/codebert-base"
    
    print(f"--- Training Model B: {args.model_name_or_path} ---")
    
    # 2. Load CodeBERT components
    config = RobertaConfig.from_pretrained(args.model_name_or_path)
    config.num_labels = 2
    tokenizer = RobertaTokenizerFast.from_pretrained(args.tokenizer_name)
    encoder = RobertaModel.from_pretrained(args.model_name_or_path, config=config)
    
    # 3. Initialize Simple Model
    model = SimpleModel(encoder, config, tokenizer, args)
    model.to(args.device)
    
    # 4. Train using the Simple Dataset
    # We copy the training loop logic but use the new classes
    dataset = SimpleCodeDataset(tokenizer, args, args.train_file)
    
    # Split
    train_size = int(0.9 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_dataset, batch_size=args.train_batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=args.eval_batch_size, num_workers=2)
    
    optimizer = AdamW(model.parameters(), lr=args.learning_rate, eps=1e-8)
    model.train()
    
    # Short training loop (Copy of previous logic, simplified)
    for epoch in range(args.num_train_epochs):
        tr_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch}"):
            inputs = {
                'input_ids': batch['input_ids'].to(args.device),
                'attention_mask': batch['attention_mask'].to(args.device),
                'labels': batch['label'].to(args.device)
            }
            optimizer.zero_grad()
            loss, _ = model(**inputs)
            loss.backward()
            optimizer.step()
            tr_loss += loss.item()
            
        # Quick Validation
        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                inputs = {
                    'input_ids': batch['input_ids'].to(args.device),
                    'attention_mask': batch['attention_mask'].to(args.device),
                    'labels': batch['label'].to(args.device)
                }
                _, prob = model(**inputs)
                val_preds.extend(torch.argmax(prob, dim=-1).cpu().numpy())
                val_labels.extend(inputs['labels'].cpu().numpy())
        
        acc = accuracy_score(val_labels, val_preds)
        print(f"Epoch {epoch} Acc: {acc:.4%}")
        model.train()

    # Save
    if not os.path.exists(args.output_dir): os.makedirs(args.output_dir)
    torch.save(model.state_dict(), os.path.join(args.output_dir, "model_codebert.bin"))
    print("Saved CodeBERT model!")

# Run it
#train_codebert()

In [10]:
import torch
import numpy as np
from torch.utils.data import DataLoader, random_split
from transformers import RobertaConfig, RobertaModel, RobertaTokenizerFast
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from tqdm import tqdm
import os

def run_ensemble_evaluation(args):
    print("\n" + "="*40)
    print("🚀 STARTING ENSEMBLE EVALUATION")
    print("="*40)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # --- 1. SETUP MODEL A (GraphCodeBERT) ---
    print("\n[1/4] Loading Model A: GraphCodeBERT...")
    path_a = "/kaggle/input/model2/model.bin"  # Update if needed
    name_a = "microsoft/graphcodebert-base"
    
    # Config & Tokenizer A
    config_a = RobertaConfig.from_pretrained(name_a)
    config_a.num_labels = 2
    tokenizer_a = RobertaTokenizerFast.from_pretrained(name_a)
    encoder_a = RobertaModel.from_pretrained(name_a, config=config_a)
    
    # Init & Load Weights
    model_a = Model(encoder_a, config_a, tokenizer_a, args)
    if os.path.exists(path_a):
        model_a.load_state_dict(torch.load(path_a, map_location=device))
        print(f"   -> Loaded weights from {path_a}")
    else:
        print(f"   -> WARNING: Model A weights not found at {path_a}!")
    model_a.to(device)
    model_a.eval()
    
    # Dataset A (Graph Mode)
    # We use the dataset class that includes DFG extraction/alignment
    full_dataset_a = TextDataset(tokenizer_a, args, args.train_file)


    # --- 2. SETUP MODEL B (CodeBERT) ---
    print("\n[2/4] Loading Model B: CodeBERT...")
    path_b = "saved_models_codebert/model_codebert.bin" # Check this path matches your save
    name_b = "microsoft/codebert-base"
    
    # Config & Tokenizer B
    config_b = RobertaConfig.from_pretrained(name_b)
    config_b.num_labels = 2
    tokenizer_b = RobertaTokenizerFast.from_pretrained(name_b)
    encoder_b = RobertaModel.from_pretrained(name_b, config=config_b)
    
    # Init & Load Weights
    model_b = SimpleModel(encoder_b, config_b, tokenizer_b, args)
    if os.path.exists(path_b):
        model_b.load_state_dict(torch.load(path_b, map_location=device))
        print(f"   -> Loaded weights from {path_b}")
    else:
        print(f"   -> WARNING: Model B weights not found at {path_b}!")
    model_b.to(device)
    model_b.eval()
    
    # Dataset B (Text Mode)
    # We use the simple dataset class (Text only)
    full_dataset_b = SimpleCodeDataset(tokenizer_b, args, args.train_file)


    # --- 3. DATA ALIGNMENT ---
    print("\n[3/4] Aligning Validation Sets...")
    # Critical: We must use the exact same seed to ensure random_split
    # selects the SAME indices for both datasets.
    generator = torch.Generator().manual_seed(args.seed)
    
    # Split A
    train_size = int(0.9 * len(full_dataset_a))
    val_size = len(full_dataset_a) - train_size
    _, val_dataset_a = random_split(full_dataset_a, [train_size, val_size], generator=generator)
    
    # Split B (Reset generator is key!)
    generator = torch.Generator().manual_seed(args.seed) 
    _, val_dataset_b = random_split(full_dataset_b, [train_size, val_size], generator=generator)
    
    # Loaders
    loader_a = DataLoader(val_dataset_a, batch_size=args.eval_batch_size, shuffle=False, num_workers=2)
    loader_b = DataLoader(val_dataset_b, batch_size=args.eval_batch_size, shuffle=False, num_workers=2)
    
    print(f"   -> Evaluating on {len(val_dataset_a)} examples.")

    
    # --- 4. PREDICTION LOOP ---
    print("\n[4/4] Running Ensemble Inference...")
    all_preds = []
    all_labels = []
    
    # Zip allows us to iterate both loaders simultaneously
    # Since we fixed the seed, batch_a[i] corresponds to the exact same file as batch_b[i]
    with torch.no_grad():
        for batch_a, batch_b in tqdm(zip(loader_a, loader_b), total=len(loader_a), desc="Ensembling"):
            
            # --- Model A Prediction ---
            inputs_a = {
                'input_ids': batch_a['input_ids'].to(device),
                'p_ids': batch_a['p_ids'].to(device),
                'attn_mask': batch_a['attn_mask'].to(device)
            }
            probs_a = model_a(**inputs_a) # Shape: [Batch, 2]
            
            # --- Model B Prediction ---
            inputs_b = {
                'input_ids': batch_b['input_ids'].to(device),
                'attention_mask': batch_b['attention_mask'].to(device)
            }
            probs_b = model_b(**inputs_b) # Shape: [Batch, 2]
            
            # --- Safety Check ---
            # Ensure labels match (sanity check that data is aligned)
            labels_a = batch_a['label'].to(device)
            labels_b = batch_b['label'].to(device)
            if not torch.equal(labels_a, labels_b):
                print("CRITICAL ERROR: Data alignment failed! Indices do not match.")
                break

            # --- ENSEMBLE LOGIC (Soft Voting) ---
            # Average the probabilities
            avg_probs = (probs_a + probs_b) / 2.0
            
            # Final Decision
            preds = torch.argmax(avg_probs, dim=-1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels_a.cpu().numpy())

    # --- REPORTING ---
    print("\n" + "="*40)
    print("🏆 FINAL ENSEMBLE RESULTS")
    print("="*40)
    
    acc = accuracy_score(all_labels, all_preds)
    print(f"Ensemble Accuracy: {acc:.4%}")
    print("-" * 40)
    print(classification_report(all_labels, all_preds, target_names=['Safe', 'Vulnerable'], digits=4))
    
    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()
    print("\nConfusion Matrix:")
    print(f"True Negatives: {tn}")
    print(f"False Positives: {fp}")
    print(f"False Negatives: {fn}")
    print(f"True Positives: {tp}")

# Run the Ensemble
#run_ensemble_evaluation(args)

In [11]:
def main():
    set_seed(args.seed)
    
    # --- Existing Setup for Model A (GraphCodeBERT) ---
    config = RobertaConfig.from_pretrained(args.model_name_or_path)
    config.num_labels = 2
    tokenizer = RobertaTokenizerFast.from_pretrained(args.tokenizer_name)
    encoder = RobertaModel.from_pretrained(args.model_name_or_path, config=config)
    
    model = Model(encoder, config, tokenizer, args)
    model.to(args.device)
    
    # --- COMMENTED OUT OLD LOGIC (As requested) ---
    # train(model, tokenizer)  
    # manual_evaluation(model, tokenizer, args) 
    
    # --- NEW ENSEMBLE LOGIC (Must be added here!) ---
    
    # 1. Train the Partner Model (CodeBERT)
    # This creates 'saved_models_codebert/model_codebert.bin'
    train_codebert()
    
    # 2. Run the Ensemble Evaluation
    # This loads both your input model (GraphCodeBERT) and the new model (CodeBERT)
    # and prints the final combined accuracy.
    run_ensemble_evaluation(args)

if __name__ == "__main__":
    main()

config.json:   0%|          | 0.00/539 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

...

Epoch 2 Acc: 88.1476%
Saved CodeBERT model!

🚀 STARTING ENSEMBLE EVALUATION

[1/4] Loading Model A: GraphCodeBERT...


Some weights of RobertaModel were not initialized from the model checkpoint at microsoft/graphcodebert-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


   -> Loaded weights from /kaggle/input/model2/model.bin

[2/4] Loading Model B: CodeBERT...
   -> Loaded weights from saved_models_codebert/model_codebert.bin

[3/4] Aligning Validation Sets...
   -> Evaluating on 19996 examples.

[4/4] Running Ensemble Inference...


Ensembling: 100%|██████████| 625/625 [09:00<00:00,  1.16it/s]



🏆 FINAL ENSEMBLE RESULTS
Ensemble Accuracy: 91.8684%
----------------------------------------
              precision    recall  f1-score   support

        Safe     0.9304    0.9068    0.9184     10095
  Vulnerable     0.9074    0.9308    0.9189      9901

    accuracy                         0.9187     19996
   macro avg     0.9189    0.9188    0.9187     19996
weighted avg     0.9190    0.9187    0.9187     19996


Confusion Matrix:
True Negatives: 9154
False Positives: 941
False Negatives: 685
True Positives: 9216
